In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
import os
import random
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Using device: {device}')

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(7)

In [ ]:
# TXT FILE PARSER 
# Reads a PROFILES txt file and extracts valid normalized Raman profiles
# Rules:
#   - Skip header lines (start with C:\)
#   - Skip index rows (1, 2, 3 ... 1060)
#   - Skip wavelength rows (137, 140, 143 ...)
#   - Skip zero rows (all zeros)
#   - Skip baseline/spike rows (values >= 3.0)
#   - Keep only first valid normalized row per block

def parse_txt_profiles(txt_path):
    profiles = []

    with open(txt_path, 'r') as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        if line.startswith('C:'):
            i += 1  # skip header
            i += 1  # skip index row
            i += 1  # skip wavelength row

            # Read rows until next block
            found = False
            while i < len(lines) and not lines[i].strip().startswith('C:'):
                row = lines[i].strip()
                if row and not found:
                    try:
                        values = [float(v.replace('+', '').replace(',', '.')) 
                                  for v in row.split('\t') if v.strip()]
                        if (len(values) == 1060 and
                            min(values) < 1.0 and
                            max(values) < 3.0 and
                            sum(values) / len(values) > 0.05):  # exclude zero rows
                            profiles.append(values)
                            found = True  # only first valid row per block
                    except ValueError:
                        pass
                i += 1
        else:
            i += 1

    return np.array(profiles, dtype=np.float32)


# DATA LOADING FROM TXT FILES 
BASE_PATH = os.path.expanduser('~/Downloads/rock_txts')

txt_files_183 = {
    'S10Granite':           os.path.join(BASE_PATH, 'PROFILES_S10Granite_1-83Hz.txt'),
    'Holstein_Sandstone':   os.path.join(BASE_PATH, 'PROFILES_HolstSandstone_1.83Hz.txt'),
    'Leitendorf_Limestone': os.path.join(BASE_PATH, 'PROFILES_Limestone_1-83Hz.txt'),
}

txt_files_510 = {
    'S10Granite':           os.path.join(BASE_PATH, 'PROFILES_S10Granite_5-10Hz.txt'),
    'Holstein_Sandstone':   os.path.join(BASE_PATH, 'PROFILES_HolstSandstone_5-10Hz.txt'),
    'Leitendorf_Limestone': os.path.join(BASE_PATH, 'PROFILES_Limestone_5-10Hz.txt'),
}

def load_data_from_txt(txt_dict):
    dfs = []
    class_names = list(txt_dict.keys())

    for label, (rock_name, path) in enumerate(txt_dict.items()):
        if not os.path.exists(path):
            print(f'[WARNING] File not found: {path}')
            continue
        profiles = parse_txt_profiles(path)
        class_num = np.full(profiles.shape[0], label, dtype=np.float32)
        d = np.c_[profiles, class_num]
        dfs.extend(d)
        print(f'{rock_name}: {profiles.shape[0]} profiles')

    dataset = np.array(dfs)
    return dataset, class_names


print('Loading 1.83 Hz data from TXT...')
dataset_183, class_names = load_data_from_txt(txt_files_183)
print(f'Total 1.83 Hz: {dataset_183.shape}\n')

print('Loading 5.10 Hz data from TXT...')
dataset_510, _ = load_data_from_txt(txt_files_510)
print(f'Total 5.10 Hz: {dataset_510.shape}')

In [ ]:
class_names

In [ ]:
# VISUALIZE RAW PROFILES FROM TXT 
X_183 = dataset_183[:, :-1]
y_183 = dataset_183[:, -1:]

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
for i, rock_name in enumerate(class_names):
    idx = np.where(y_183.flatten() == i)[0][:5]
    for j in idx:
        axes[i].plot(X_183[j], alpha=0.6)
    axes[i].set_title(f'{rock_name} - 1.83 Hz')
    axes[i].set_xlabel('Wavelength index')
    axes[i].set_ylabel('Intensity')

plt.suptitle('Raman Profiles - 1.83 Hz (from TXT)', fontsize=14)
plt.tight_layout()
_RES = 'results_1d_cnn_raw_txt'
import os as _os; _os.makedirs(_RES, exist_ok=True)
_p = _os.path.join(_RES, 'raw_profiles_183Hz.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
print(f'[SAVED] raw_profiles_183Hz.png -> {_p}')
plt.show()

In [ ]:
X_510 = dataset_510[:, :-1]
y_510 = dataset_510[:, -1:]

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
for i, rock_name in enumerate(class_names):
    idx = np.where(y_510.flatten() == i)[0][:5]
    for j in idx:
        axes[i].plot(X_510[j], alpha=0.6)
    axes[i].set_title(f'{rock_name} - 5.10 Hz')
    axes[i].set_xlabel('Wavelength index')
    axes[i].set_ylabel('Intensity')

plt.suptitle('Raman Profiles - 5.10 Hz (from TXT)', fontsize=14)
plt.tight_layout()
_RES = 'results_1d_cnn_raw_txt'
import os as _os; _os.makedirs(_RES, exist_ok=True)
_p = _os.path.join(_RES, 'raw_profiles_510Hz.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
print(f'[SAVED] raw_profiles_510Hz.png -> {_p}')
plt.show()

In [ ]:
print(f'X_183: {X_183.shape}, y_183: {y_183.shape}')
print(f'X_510: {X_510.shape}, y_510: {y_510.shape}')

In [ ]:
X_train_183, X_test_183, y_train_183, y_test_183 = sklearn.model_selection.train_test_split(
    X_183, y_183, test_size=0.2, random_state=3, stratify=y_183
)
X_train_510, X_test_510, y_train_510, y_test_510 = sklearn.model_selection.train_test_split(
    X_510, y_510, test_size=0.2, random_state=3, stratify=y_510
)

print('1.83 Hz -> Train:', X_train_183.shape, ' Test:', X_test_183.shape)
print('5.10 Hz -> Train:', X_train_510.shape, ' Test:', X_test_510.shape)

In [ ]:
class RockDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

    def __len__(self):
        return len(self.X)


train_dataset_183 = RockDataset(X_train_183, y_train_183)
test_dataset_183  = RockDataset(X_test_183,  y_test_183)

train_dataset_510 = RockDataset(X_train_510, y_train_510)
test_dataset_510  = RockDataset(X_test_510,  y_test_510)

In [ ]:
import itertools

batch_sizes   = [2048]
lrs           = [0.0001, 0.00001]
epochs_list   = [100]
weight_decays = [0., 1e-4]

hyperparameters_list = []
for batch_size, lr, epochs, weight_decay in itertools.product(batch_sizes, lrs, epochs_list, weight_decays):
    hyperparameters_list.append({
        'batch_size'  : batch_size,
        'lr'          : lr,
        'epochs'      : epochs,
        'weight_decay': weight_decay
    })

print(f'Generated {len(hyperparameters_list)} combinations.')

In [ ]:
# MODEL (3 conv layers) 
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm_input = nn.LayerNorm(normalized_shape=1060)
        self.conv1 = nn.Conv1d(1,  16, kernel_size=9, stride=3)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=9, stride=3)
        self.conv3 = nn.Conv1d(32, 64, kernel_size=5, stride=2)
        self.layer_norm_1 = nn.LayerNorm(normalized_shape=64)
        self.fnn1 = nn.Linear(64, len(class_names))

        self.relu        = nn.ReLU()
        self.max_pool    = nn.MaxPool1d(kernel_size=2)
        self.global_pool = nn.AdaptiveMaxPool1d(output_size=1)
        self.flatten     = nn.Flatten()
        self.dropout     = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.layer_norm_input(x)
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x))
        x = self.max_pool(x)
        x = self.relu(self.conv2(x))
        x = self.max_pool(x)
        x = self.relu(self.conv3(x))
        x = self.global_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.layer_norm_1(x)
        x = self.fnn1(x)
        return x


model = Model()
print(f'Model parameters: {sum(p.numel() for p in model.parameters())}')

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import json


def run_experiments(train_dataset, test_dataset, X_test, y_test, speed_tag):

    experiment_results = []
    results_dir = f'results_1d_cnn_raw_txt/{speed_tag.replace(" ", "_")}'
    os.makedirs(results_dir, exist_ok=True)

    monitor_indices = np.arange(4)
    X_monitor = torch.from_numpy(X_test[monitor_indices]).to(device)
    y_monitor = torch.from_numpy(y_test[monitor_indices]).squeeze().long().to(device)

    print(f'Monitor True Labels: {[class_names[int(i)] for i in y_monitor]}')

    for idx, config in enumerate(hyperparameters_list):
        print(f"\n{'='*20}\nStarting Experiment {idx+1}\nConfig: {config}\n{'='*20}")

        train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True,  num_workers=0)
        test_loader  = DataLoader(test_dataset,  batch_size=config['batch_size'], shuffle=False, num_workers=0)

        model = Model().to(device)
        optimizer = torch.optim.Adam(params=model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
        criterion = nn.CrossEntropyLoss()

        best_accuracy   = 0.0
        best_model_path = os.path.join(results_dir, f'best_model_exp_{idx+1}.pth')
        tb_log_dir      = os.path.join(results_dir, f'tb_exp{idx+1}_lr{config["lr"]}_wd{config["weight_decay"]}')
        writer          = SummaryWriter(log_dir=tb_log_dir)

        training_loss = []
        training_acc  = []
        testing_loss  = []
        testing_acc   = []

        for epoch in range(config['epochs']):
            avg_train_loss = []
            avg_train_acc  = []
            avg_test_loss  = []
            avg_test_acc   = []

            print(f'Epoch: {epoch + 1}/{config["epochs"]}')

            # Train
            model.train()
            for i, d in tqdm(enumerate(train_loader), total=len(train_loader), leave=False):
                X_batch, y_batch = d
                X_batch = X_batch.to(device)
                y_batch = y_batch.squeeze().long().to(device)
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss    = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

                _, predicted_labels = torch.max(outputs, 1)
                accuracy = (predicted_labels == y_batch).sum().item() / y_batch.size(0)
                avg_train_loss.append(loss.item())
                avg_train_acc.append(accuracy)

            avg_acc_train  = sum(avg_train_acc)  / len(avg_train_acc)
            avg_loss_train = sum(avg_train_loss) / len(avg_train_loss)
            training_acc.append(avg_acc_train)
            training_loss.append(avg_loss_train)
            print(f'Training: Loss={avg_loss_train:.4f}, Acc={avg_acc_train:.4f}')

            # Test
            model.eval()
            all_preds, all_true = [], []
            with torch.no_grad():
                for i, d in tqdm(enumerate(test_loader), total=len(test_loader), leave=False):
                    X_batch, y_batch = d
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.squeeze().long().to(device)
                    outputs = model(X_batch)
                    loss    = criterion(outputs, y_batch)

                    preds = torch.max(outputs, 1)[1]
                    all_preds.extend(preds.cpu().numpy())
                    all_true.extend(y_batch.cpu().numpy())

                    acc = (preds == y_batch).sum().item() / y_batch.size(0)
                    avg_test_loss.append(loss.item())
                    avg_test_acc.append(acc)

            avg_acc_test  = sum(avg_test_acc)  / len(avg_test_acc)
            avg_loss_test = sum(avg_test_loss) / len(avg_test_loss)
            testing_acc.append(avg_acc_test)
            testing_loss.append(avg_loss_test)
            print(f'Testing: Loss={avg_loss_test:.4f}, Acc={avg_acc_test:.4f}\n')

            writer.add_scalars('Loss',     {'train': avg_loss_train, 'test': avg_loss_test}, epoch)
            writer.add_scalars('Accuracy', {'train': avg_acc_train,  'test': avg_acc_test},  epoch)

            if avg_acc_test > best_accuracy:
                best_accuracy = avg_acc_test
                torch.save(model.state_dict(), best_model_path)
                print(f' New Best Acc: {best_accuracy:.4f} - Saved Model.')

            model.eval()
            with torch.no_grad():
                outputs_monitor = model(X_monitor)
                _, pred_monitor = torch.max(outputs_monitor, 1)
                print(f'Monitor Profiles (Epoch {epoch + 1}):')
                for j, (true_cls, pred_cls) in enumerate(zip(y_monitor, pred_monitor)):
                    status = '\u2713' if true_cls == pred_cls else '\u2717'
                    print(f'  Sample {monitor_indices[j]}: True: {class_names[true_cls.item()]:<15} | Pred: {class_names[pred_cls.item()]:<15} {status}')
            print('-' * 30)

        # Confusion Matrix
        model.load_state_dict(torch.load(best_model_path, map_location=device))
        model.to(device)
        model.eval()

        all_preds, all_true = [], []
        with torch.no_grad():
            for i, d in tqdm(enumerate(test_loader), total=len(test_loader), leave=False):
                X_batch, y_batch = d
                X_batch = X_batch.to(device)
                y_batch = y_batch.squeeze().long().to(device)
                outputs = model(X_batch)
                preds   = torch.max(outputs, 1)[1]
                all_preds.extend(preds.cpu().numpy())
                all_true.extend(y_batch.cpu().numpy())

        cm = confusion_matrix(all_true, all_preds)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Confusion Matrix - {speed_tag} - Exp {idx+1}')
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.tight_layout()
        cm_path = os.path.join(results_dir, f'confusion_matrix_exp_{idx+1}.png')
        plt.savefig(cm_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'[SAVED] Confusion matrix -> {cm_path}')
        plt.close()

        writer.close()

        experiment_results.append({
            'config'         : config,
            'final_train_acc': training_acc[-1],
            'final_test_acc' : testing_acc[-1],
            'final_test_loss': testing_loss[-1],
            'best_model_path': best_model_path,
            'history': {
                'train_loss': training_loss,
                'train_acc' : training_acc,
                'test_loss' : testing_loss,
                'test_acc'  : testing_acc
            }
        })

    print('\n' + '='*40)
    print(f'RESULTS - {speed_tag}')
    print('='*40)
    for res in experiment_results:
        print(f"Config: {res['config']}")
        print(f"  -> Final Test Acc: {res['final_test_acc']:.4f}")
        print(f"  -> Final Test Loss: {res['final_test_loss']:.4f}")
        print('-' * 20)

    return experiment_results


print('\n' + '#'*50)
print('TRAINING - 1.83 Hz (from TXT)')
print('#'*50)
results_183 = run_experiments(train_dataset_183, test_dataset_183, X_test_183, y_test_183, '1.83 Hz')

print('\n' + '#'*50)
print('TRAINING - 5.10 Hz (from TXT)')
print('#'*50)
results_510 = run_experiments(train_dataset_510, test_dataset_510, X_test_510, y_test_510, '5.10 Hz')

In [ ]:
# RESULTS - 1.83 Hz 
best_183 = max(results_183, key=lambda x: x['final_test_acc'])

plt.figure(figsize=(7, 4))
plt.plot(best_183['history']['train_loss'], color='blue', label='Training loss')
plt.plot(best_183['history']['test_loss'],  color='red',  label='Testing loss')
plt.title('Loss Curves - 1.83 Hz (Best Config)')
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(best_183['history']['train_acc'], color='blue', label='Training acc')
plt.plot(best_183['history']['test_acc'],  color='red',  label='Testing acc')
plt.title('Accuracy Curves - 1.83 Hz (Best Config)')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true, y_pred = [], []
test_loader = DataLoader(test_dataset_183, batch_size=2048, shuffle=False, num_workers=0)
model = Model().to(device)
model.load_state_dict(torch.load(best_183['best_model_path'], map_location=device))
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs      = model(X_batch.to(device))
        _, predicted = torch.max(outputs, 1)
        y_true.extend(y_batch.squeeze().tolist())
        y_pred.extend(predicted.cpu().tolist())

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical')
plt.title(f'Confusion Matrix - 1.83 Hz (Best: {best_183["config"]})')
plt.show()
print(f'Best Test Acc: {best_183["final_test_acc"]:.4f}')

In [ ]:
# RESULTS - 5.10 Hz 
best_510 = max(results_510, key=lambda x: x['final_test_acc'])

plt.figure(figsize=(7, 4))
plt.plot(best_510['history']['train_loss'], color='blue', label='Training loss')
plt.plot(best_510['history']['test_loss'],  color='red',  label='Testing loss')
plt.title('Loss Curves - 5.10 Hz (Best Config)')
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(best_510['history']['train_acc'], color='blue', label='Training acc')
plt.plot(best_510['history']['test_acc'],  color='red',  label='Testing acc')
plt.title('Accuracy Curves - 5.10 Hz (Best Config)')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true, y_pred = [], []
test_loader = DataLoader(test_dataset_510, batch_size=2048, shuffle=False, num_workers=0)
model = Model().to(device)
model.load_state_dict(torch.load(best_510['best_model_path'], map_location=device))
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs      = model(X_batch.to(device))
        _, predicted = torch.max(outputs, 1)
        y_true.extend(y_batch.squeeze().tolist())
        y_pred.extend(predicted.cpu().tolist())

cm   = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical')
plt.title(f'Confusion Matrix - 5.10 Hz (Best: {best_510["config"]})')
plt.show()
print(f'Best Test Acc: {best_510["final_test_acc"]:.4f}')

In [ ]:
# SAVE BEST MODELS 
import json

for tag, results, X_tr, y_tr in [
    ('1.83Hz', results_183, X_train_183, y_train_183),
    ('5.10Hz', results_510, X_train_510, y_train_510)
]:
    best_result = max(results, key=lambda x: x['final_test_acc'])
    best_config = best_result['config']
    print(f"Best Config ({tag}): {best_config} — Acc: {best_result['final_test_acc']:.4f}")

    full_train_dataset = RockDataset(X_tr, y_tr)
    train_loader = DataLoader(full_train_dataset, batch_size=best_config['batch_size'], shuffle=True, num_workers=0)

    model = Model().to(device)
    optimizer = torch.optim.Adam(params=model.parameters(), lr=best_config['lr'], weight_decay=best_config['weight_decay'])
    criterion = nn.CrossEntropyLoss()

    for epoch in range(best_config['epochs']):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.squeeze().long().to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

    os.makedirs('results_1d_cnn_raw_txt', exist_ok=True)
    torch.save(model.state_dict(), f'results_1d_cnn_raw_txt/1d_cnn_raw_txt_{tag}.pth')
    with open(f'results_1d_cnn_raw_txt/class_names_cnn_txt_{tag}.json', 'w') as f:
        json.dump(class_names, f)
    print(f'Saved rock_classifier_cnn_txt_{tag}.pth\n')

In [ ]:
print('To view TensorBoard, run in terminal:')
print('  cd ~/results_1d_cnn_raw_txt/1.83_Hz && tensorboard --logdir=.')
print('or')
print('  cd ~/results_1d_cnn_raw_txt/5.10_Hz && tensorboard --logdir=.')
print()
print('Then open: http://localhost:6006')